In [4]:
import pdfplumber
from google import genai
from google.genai import types
import pandas as pd
import google.generativeai as genai
import json
import os.path

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def get_pdf_data1(first_page, last_page):
    pdf_path = r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\2026-05 Market Share Report Mei.pdf"
    # Read PDF and convert directly to a list of DataFrames
    all_rows = []
    headers = None

    with pdfplumber.open(pdf_path) as pdf:
        # Python ranges are exclusive at the end, so range(2, 13) targets pages 3 to 13
        for page_num in range(first_page, last_page):
            page = pdf.pages[page_num]
            tables = page.extract_tables()

            if tables:
                for table in tables:
                    # If it's the very first table we find, grab the header
                    if headers is None:
                        headers = table[0]
                        # Append the rest of the rows
                        all_rows.extend(table[1:])
                    else:
                        # For subsequent pages, skip the header row and just append data
                        # Check if the header matches roughly to avoid misaligned data
                        if table[0] == headers:
                            all_rows.extend(table[1:])
                        else:
                            all_rows.extend(table) # Append all if headers are slightly different

    # Convert the giant list of rows into a single DataFrame
    if all_rows and headers:
        df_combined = pd.DataFrame(all_rows, columns=headers)
        print(f"Successfully extracted data. Total rows: {len(df_combined)}")

        # Clean up empty rows or rows that might have duplicated headers
        df_combined = df_combined.dropna(how='all')
        #print(df_combined)
        display(df_combined.head(12))
        return df_combined
    else:
        print("No tables found in the specified page range.")
        return None

def ai_processing(raw_df):
    genai.configure(api_key="AQ.Ab8RN6I7WqbeUlRXPjXm9C75hCweJNo5M8AOkZgJ2HP2atp1WA")
    generation_config = {
        "temperature": 0,
        "top_p": 0.95,
        "response_mime_type": "application/json",
    }
    model = genai.GenerativeModel(
      model_name="gemini-3.5-flash",
      generation_config=generation_config,
    )
    prompt = f"""
            Act as a precise data extraction engine. Extract data metrics from the provided raw data dump.

            ### EXTRACTION RULES:
            1. MAPPING: Locate the exact row for each of the province.
            2. VALUES: 
               - If a value is missing, null, or "-" in the raw data, return 0.0. 
               - DO NOT use nested dictionaries. Use a flat structure.
               - Fill Month Value with Mei.
            3. For every province group it to 
            4. Fill the missing value in PROJECT CODE / WBS or NAMA CAPEX with "Belum terisi"
            5. FORMAT: Return ONLY a valid JSON list of objects. No conversational text.

            
            ### REQUIRED JSON STRUCTURE EXAMPLE:
            [
              {{
                "1. Tahun Capex": 2026,
                "2. Item Capex": "IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM 5 DAN CM 9 GHOPO TUBAN ",
                "3. No. Project / WBS": "P6-22270-07-C-12", 
                .....
                "32. No PR": "6200004449, 6200004498", 
                "33. Total PO": 583.925.000,
                "34. No PO": "7400003139, 7400003158",
                "37. Spending / GR / SA tahun sebelumnya": 500.000.000
              }},
              {{
                "1. Tahun Capex": 2026,
                "2. Item Capex": "SP-T REPLACE RBC RH A2J02",
                "3. No. Project / WBS": "P2-26107-03-C-07", 
                .....
                "32. No PR": "6200004449, 6200004498", 
                "33. Total PO": 583.925.000,
                "34. No PO": "7400003139, 7400003158",
                "37. Spending / GR / SA tahun sebelumnya": 0
              }}
            ]

            Raw Data:
            {raw_data_string}
            """# 4. Generate response
    response = model.generate_content(
        prompt,
        request_options={"timeout": 600.0}  # 10 minutes
    )
    json_text = response.text.replace('```json', '').replace('```', '').strip()

    print(f"DEBUG: Response Length: {len(json_text)}") # See if it's hitting a limit
    try:
        data = json.loads(json_text)
        # 5. Convert back to DataFrame
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None

def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def update_sheet(spreadsheetId, spread_range, scopes, creds, df):
    df_clean = df.fillna('')
    header = df.columns.tolist()
    data = df_clean.values.tolist()
    
    # Combine header and rows so the column titles are updated too
    values_with_header = [header] + data
    
    try:
        service = build('sheets', 'v4', credentials=creds)
        
        # 1. Clear the existing data in the range first so old data doesn't linger
        service.spreadsheets().values().clear(
            spreadsheetId=spreadsheetId,
            range=spread_range
        ).execute()
        
        # 2. Update with the new complete dataset
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId,
            range=spread_range,
            valueInputOption="USER_ENTERED",
            body=body
        ).execute()
        
        print(f"{result.get('updatedCells')} cells updated successfully (overwritten).")
        return result
        
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
def main():
    raw_df = get_pdf_data1(2, 13)
    df = raw_df
    #df = ai_processing(raw_df)
    if df is not None and not df.empty:
        print("Data extracted successfully.")
        
        creds = connect_to_sheet()
        
        # Append Raw Data
        update_sheet('1r_2BqGqW3_IR4eJZTGUKe2CDV7TPiYWudozillg3uck', 'Sheet1', None, creds, df)
        
    else:
        print("Failed to extract data. Check Gemini response or JSON format.")
if __name__ == '__main__':
    main()

Successfully extracted data. Total rows: 55


,Province,Affordability,Attractiveness,Availability,Advocacy,Insight,Recommendation
0,"Aceh\n%MS (MoM)\nSIG: 89,4 (-2,2)\nADLS: 38,9 ...",,,▪ Penurunan DYX dipengaruhi oleh\nketerbatasan...,,Kendala distribusi yang masih dialami MP\npasc...,▪ Prioritaskan pemenuhan supply dan percepatan...
1,"Sumut\n%MS (MoM)\nSIG: 54,2 (-1,3)\nADLS: 3,7 ...",,SMP menjalankan program\nfree gift minyak gore...,▪ SIG mengalami kendala supply akibat stock\no...,,Pelemahan INTP seharusnya menjadi\npeluang bag...,▪ Optimalkan program marketing berjalan (SIG A...
2,"Sumbar\n%MS (MoM)\nSIG: 91,7 (+0,2)\nSP: 81,7 ...",Kenaikan harga TR yang mulai\nefektif pada bul...,,,SIG melakukan\nTemu Jago\nBangunan di\nSolok (...,Kenaikan harga TR membuka peluang\nbagi SIG un...,▪ Optimalkan program marketing berjalan (SIG A...
3,"Riau\n%MS (MoM)\nSIG: 57,4 (-2,3)\nSP: 43,6 (+...",Kenaikan harga kompetitor TR\ndan MP mulai ber...,▪ SMP menjalankan\nprogram cashback\nRp500/sak...,▪ Penurunan MS SIG dipengaruhi oleh kendala\na...,,Pelemahan INTP akibat keterbatasan\nsupply bel...,▪ Program push sell out sebagai upaya meng-cou...
4,"Rikep\n%MS (MoM)\nSIG: 29,2 (+11,3)\nADLS: 18,...",Kenaikan harga oleh competitor\nkarena kenaika...,▪ TR program cashback 1.000/sak\nminimal order...,▪ RJWL dan TR sempat mengalami\nkekosongan sto...,,Kenaikan harga kompetitor dan keterbatasan\nsu...,▪ Optimalkan program existing untuk\nmenjaga m...
5,"Bengkulu\n%MS (MoM)\nSIG: 71,7 (+9,2)\nSP: 46,...",Kenaikan harga MP sebesar\n1.500/sak mulai efe...,,Pertumbuhan brand DYX krn fluktuasi\nkapal yan...,,▪ Pertumbuhan MS SIG di Bengkulu\ndidukung ole...,▪ Optimalkan program existing untuk\nmengakuis...
6,"Jambi\n%MS (MoM)\nSIG: 52,2 (+3,7)\nSP: 49,9 (...",MP melakukan kenaikan harga\nsebesar Rp2.000/s...,,TR mengalami kendala supply pasca\nkenaikan ta...,,▪ Pertumbuhan MS SIG di Jambi didukung\noleh p...,▪ Optimalkan program kontraktual\nuntuk menang...
7,"Sumsel\n%MS (MoM)\nSIG: 58,7 (+7,7)\nSMBR: 44,...",Kondisi kenaikan harga kompetitor\nINTP mulai ...,Brand visibility SIG rendah\ndibandingkan comp...,▪ Supply SIG juga terkendala\nantrian truk dip...,▪ Belum terjalin kedekatan antara\ntoko dan sa...,Meskipun kinerja MS Sumsel di bulan Mei\nmenca...,▪ Optimalisasi program Marketing\nPromo berjal...
8,"Lampung\n%MS (MoM)\nSIG: 53,8 (+2,0)\nSMBR: 29...",Kenaikan harga kompetitor MP mulai\nberlaku ef...,"Area Pringsewu, Metro,\nLampung Utara, Tulang\...",Kendala Credit Limit distributor SIG\nmenyebab...,,▪ Agresifitas program kompetitor perlu\nterus ...,▪ Optimalisasi program Marketing\nPromo berjal...
9,Insight & Rekomendasi AP 3 Retail (1/2)\nProvi...,None,None,None,None,None,None


Data extracted successfully.
392 cells updated successfully (overwritten).
